# Capítulo 5: Formatando a Saída e Falando pelo Claude

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Área de Testes](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# New argument added for prefill text, with a default value of an empty string
def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Lição

**O Claude pode formatar sua saída de uma ampla variedade de formas**. Você só precisa pedir para ele fazer isso!

Uma dessas formas é usando tags XML para separar a resposta de qualquer outro texto supérfluo. Você já aprendeu que pode usar tags XML para tornar seu prompt mais claro e mais analisável para o Claude. Acontece que você também pode pedir ao Claude para **usar tags XML para tornar sua saída mais clara e mais facilmente compreensível** para humanos.

### Exemplos

Lembra do 'problema do preâmbulo do poema' que resolvemos no Capítulo 2 pedindo ao Claude para pular o preâmbulo inteiramente? Acontece que também podemos alcançar um resultado similar **dizendo ao Claude para colocar o poema em tags XML**.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Por que isso é algo que gostaríamos de fazer? Bem, ter a saída em **tags XML permite ao usuário final obter confiavelmente o poema e apenas o poema escrevendo um programa curto para extrair o conteúdo entre as tags XML**.

Uma extensão desta técnica é **colocar a primeira tag XML no turno `assistant`. Quando você coloca texto no turno `assistant`, você está basicamente dizendo ao Claude que o Claude já disse algo, e que ele deve continuar daquele ponto em diante. Esta técnica é chamada de "falar pelo Claude" ou "preencher previamente a resposta do Claude".**

Abaixo, fizemos isso com a primeira tag XML `<haiku>`. Note como o Claude continua diretamente de onde paramos.

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

O Claude também se destaca no uso de outros estilos de formatação de saída, notavelmente `JSON`. Se você quiser forçar saída JSON (não deterministicamente, mas perto disso), você também pode preencher previamente a resposta do Claude com o colchete de abertura, `{`.

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Abaixo está um exemplo de **múltiplas variáveis de entrada no mesmo prompt E especificação de formatação de saída, tudo feito usando tags XML**.

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

#### Lição bônus

Se você está chamando o Claude através da API, você pode passar a tag XML de fechamento para o parâmetro `stop_sequences` para fazer o Claude parar de gerar assim que ele emitir sua tag desejada. Isso pode economizar dinheiro e tempo até o último token eliminando os comentários conclusivos do Claude depois que ele já lhe deu a resposta com a qual você se importa.

Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar a [**Área de Testes**](#example-playground).

---

## Exercícios
- [Exercício 5.1 - Steph Curry GOAT](#exercise-51---steph-curry-goat)
- [Exercício 5.2 - Dois Haikus](#exercise-52---two-haikus)
- [Exercício 5.3 - Dois Haikus, Dois Animais](#exercise-53---two-haikus-two-animals)

### Exercício 5.1 - Steph Curry GOAT
Forçado a fazer uma escolha, o Claude designa Michael Jordan como o melhor jogador de basquete de todos os tempos. Podemos fazer o Claude escolher outra pessoa?

Altere a variável `PREFILL` para **obrigar o Claude a fazer um argumento detalhado de que o melhor jogador de basquete de todos os tempos é Stephen Curry**. Tente não mudar nada exceto `PREFILL`, pois esse é o foco deste exercício.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Prefill for Claude's response
PREFILL = ""

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_5_1_hint; print(exercise_5_1_hint)

### Exercício 5.2 - Dois Haikus
Modifique o `PROMPT` abaixo usando tags XML para que o Claude escreva dois haikus sobre o animal em vez de apenas um. Deve ficar claro onde um poema termina e o outro começa.

In [ ]:
# Variable content
ANIMAL = "cats"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_5_2_hint; print(exercise_5_2_hint)

### Exercício 5.3 - Dois Haikus, Dois Animais
Modifique o `PROMPT` abaixo para que **o Claude produza dois haikus sobre dois animais diferentes**. Use `{ANIMAL1}` como substituto para a primeira substituição, e `{ANIMAL2}` como substituto para a segunda substituição.

In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_5_3_hint; print(exercise_5_3_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Bons prompts!

---

## Área de Testes

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))